# Capstone 9.1 — A Data Pipeline

### Python Mastery · Track 9: Capstone Projects

This capstone integrates several earlier tracks into one realistic build: a **data pipeline** that ingests raw files, cleans and transforms the data, stores it in a database, and produces a report. It draws on file handling and data structures (Track 2), databases (Module 8.1), the data stack (Module 8.5), error handling, and testing (Track 6).

**How to use this notebook**

- Read each section, then run the code cell beneath it (`Shift + Enter`). The pipeline is built in stages and runs end to end on generated sample data, so nothing external is required.
- The final section assembles the pieces into a real project layout you can take away and run as a script.
- Extension challenges appear near the end, with solutions, so you can take the project further.

### What you will build

A pipeline with four clear stages, **extract**, **transform**, **load**, **report** (the classic ETL shape plus reporting), wired together with validation and logging:

1. **Extract:** read sales records from a CSV file and product details from a JSON file.
2. **Transform:** clean the data, join the two sources, and derive new fields.
3. **Load:** write the cleaned, joined data into a SQLite database.
4. **Report:** query the database to produce a summary.

### Learning objectives

After completing this capstone you will be able to:

1. Structure a multi-stage pipeline with separate, testable functions.
2. Read and validate data from CSV and JSON sources.
3. Clean and join data with Pandas, handling missing and malformed values.
4. Persist results to SQLite and query them for a report.
5. Add logging and a test for a data-processing component.

**Prerequisites:** Tracks 1 to 6, and Modules 8.1 and 8.5.

---

## Stage 0 · Generate Sample Data

A pipeline needs input. To keep the notebook self-contained and reproducible, we generate a small, deliberately **messy** dataset: a CSV of sales with some missing and badly typed values, and a JSON file of product information. Real pipelines face exactly this kind of imperfect input, so cleaning it is part of the job.

In [ ]:
import csv, json, os

os.makedirs("pipeline_data", exist_ok=True)

# A sales CSV with intentional problems: a missing quantity, a blank price,
# whitespace, inconsistent casing, and an unknown product id.
sales_rows = [
    {"order_id": "1", "product_id": "P1", "quantity": "2",  "unit_price": "1.50", "region": "North"},
    {"order_id": "2", "product_id": "P2", "quantity": "1",  "unit_price": "4.00", "region": "south"},
    {"order_id": "3", "product_id": "P1", "quantity": "",    "unit_price": "1.50", "region": "North"},  # missing qty
    {"order_id": "4", "product_id": "P3", "quantity": "5",  "unit_price": "",     "region": "East"},   # missing price
    {"order_id": "5", "product_id": "P2", "quantity": "3",  "unit_price": "4.00", "region": " South "}, # whitespace/case
    {"order_id": "6", "product_id": "P9", "quantity": "1",  "unit_price": "9.99", "region": "West"},   # unknown product
]
with open("pipeline_data/sales.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["order_id", "product_id", "quantity", "unit_price", "region"])
    writer.writeheader()
    writer.writerows(sales_rows)

# Product details as JSON.
products = {
    "P1": {"name": "Pen", "category": "Stationery"},
    "P2": {"name": "Notebook", "category": "Stationery"},
    "P3": {"name": "Lamp", "category": "Home"},
}
with open("pipeline_data/products.json", "w") as f:
    json.dump(products, f, indent=2)

print("sample data written to pipeline_data/")
print("sales.csv rows:", len(sales_rows), "| products:", list(products))

## Stage 1 · Extract

The first stage reads the raw sources into memory. Each reader is a small function that does one thing, which makes the pipeline easy to test and reason about. We read the CSV with Pandas (which gives us a DataFrame to clean) and the JSON with the standard library. Reading is also where we attach **logging**, so a run leaves a trace of what happened.

In [ ]:
import logging
import pandas as pd
import json

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
log = logging.getLogger("pipeline")

def extract_sales(csv_path):
    """Read the sales CSV into a DataFrame, keeping all values as strings for now."""
    df = pd.read_csv(csv_path, dtype=str)        # read as text; we will clean and convert next
    log.info(f"extracted {len(df)} sales rows from {csv_path}")
    return df

def extract_products(json_path):
    """Read the product details JSON into a dictionary."""
    with open(json_path) as f:
        products = json.load(f)
    log.info(f"extracted {len(products)} products from {json_path}")
    return products

sales_df = extract_sales("pipeline_data/sales.csv")
products = extract_products("pipeline_data/products.json")
print(sales_df)

## Stage 2 · Transform

The heart of the pipeline. Here we clean the messy input and enrich it:

- Normalise the `region` text (strip whitespace, consistent casing).
- Convert `quantity` and `unit_price` to numbers, coercing blanks to missing values.
- Drop rows that cannot be used (missing essential numbers).
- Join in the product name and category, and flag sales whose product is unknown.
- Derive a `revenue` column.

Each step is explicit, so the cleaning rules are visible and auditable, which matters in real pipelines where data quality decisions need to be justified.

In [ ]:
import pandas as pd
import numpy as np

def transform(sales_df, products):
    """Clean, join, and enrich the sales data; return a tidy DataFrame."""
    df = sales_df.copy()

    # Normalise region: strip surrounding whitespace and use title case.
    df["region"] = df["region"].str.strip().str.title()

    # Convert numeric columns; blanks and bad values become NaN (missing).
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

    # Record how many rows are incomplete before dropping them.
    incomplete = df[df["quantity"].isna() | df["unit_price"].isna()]
    log.info(f"dropping {len(incomplete)} rows with missing quantity or price")
    df = df.dropna(subset=["quantity", "unit_price"]).copy()

    # Join product details. Map by product_id; unknown ids get a placeholder.
    df["product_name"] = df["product_id"].map(lambda pid: products.get(pid, {}).get("name", "UNKNOWN"))
    df["category"] = df["product_id"].map(lambda pid: products.get(pid, {}).get("category", "UNKNOWN"))

    unknown = df[df["product_name"] == "UNKNOWN"]
    if len(unknown):
        log.warning(f"{len(unknown)} sales reference an unknown product id")

    # Derive revenue. quantity is now a clean float; cast to int for tidiness.
    df["quantity"] = df["quantity"].astype(int)
    df["revenue"] = df["quantity"] * df["unit_price"]

    return df.reset_index(drop=True)

clean_df = transform(sales_df, products)
print(clean_df[["order_id", "product_name", "region", "quantity", "unit_price", "revenue"]])

The transform stage has done real work: the row with a missing quantity and the row with a missing price were dropped, the " South " value was normalised to "South" and merged with the other South rows, product names and categories were joined in, and the sale referencing the unknown product `P9` was flagged in the log and labelled `UNKNOWN` rather than silently discarded. This is the kind of explicit, defensible cleaning a production pipeline needs.

## Stage 3 · Load

With clean data in hand, we persist it to a database so it can be queried and shared, the lesson from Module 8.1. We write the DataFrame to a SQLite table. Pandas can write directly to SQLite via a connection, which is convenient; we also show that the data is queryable afterwards.

In [ ]:
import sqlite3

def load(df, db_path, table="sales"):
    """Write the cleaned DataFrame into a SQLite table, replacing any existing one."""
    conn = sqlite3.connect(db_path)
    df.to_sql(table, conn, if_exists="replace", index=False)
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    conn.close()
    log.info(f"loaded {count} rows into {db_path}::{table}")
    return count

rows_loaded = load(clean_df, "pipeline_data/warehouse.db")
print("rows in the database:", rows_loaded)

# Confirm the data is queryable.
conn = sqlite3.connect("pipeline_data/warehouse.db")
sample = conn.execute("SELECT order_id, product_name, revenue FROM sales LIMIT 3").fetchall()
conn.close()
print("sample rows:", sample)

## Stage 4 · Report

The final stage answers questions from the stored data. Because the data is in SQLite, we can use SQL aggregation (Module 8.1) for the report, or Pandas, whichever is clearer. Here we produce two summaries: revenue by region and revenue by product category. A real pipeline would write these to a file or dashboard; we print them.

In [ ]:
import sqlite3
import pandas as pd

def report(db_path):
    """Produce summary tables from the warehouse database."""
    conn = sqlite3.connect(db_path)

    by_region = pd.read_sql_query(
        "SELECT region, SUM(revenue) AS total_revenue, COUNT(*) AS orders "
        "FROM sales GROUP BY region ORDER BY total_revenue DESC",
        conn,
    )
    by_category = pd.read_sql_query(
        "SELECT category, SUM(revenue) AS total_revenue "
        "FROM sales GROUP BY category ORDER BY total_revenue DESC",
        conn,
    )
    conn.close()
    return by_region, by_category

by_region, by_category = report("pipeline_data/warehouse.db")
print("Revenue by region:")
print(by_region.to_string(index=False))
print()
print("Revenue by category:")
print(by_category.to_string(index=False))

## Putting It Together: One Pipeline Function

The stages are separate functions, which is good for testing, but a user wants to run the whole thing with one call. We compose them into a single `run_pipeline` that wires the stages together and returns the report. This is the pipeline's public entry point.

In [ ]:
def run_pipeline(csv_path, json_path, db_path):
    """Run the full extract-transform-load-report pipeline and return the summaries."""
    log.info("pipeline starting")
    sales_df = extract_sales(csv_path)
    products = extract_products(json_path)
    clean = transform(sales_df, products)
    load(clean, db_path)
    summaries = report(db_path)
    log.info("pipeline finished")
    return summaries

region_summary, category_summary = run_pipeline(
    "pipeline_data/sales.csv",
    "pipeline_data/products.json",
    "pipeline_data/warehouse.db",
)
print()
print("Top region:", region_summary.iloc[0]["region"],
      "with revenue", region_summary.iloc[0]["total_revenue"])

## Testing a Pipeline Component

Pipelines must be trustworthy, so the cleaning logic deserves a test (Track 6). The `transform` stage is the riskiest part, so we test it directly: given a tiny input with a known problem, does it produce the expected clean output? We write a small pytest file and run it. Testing transform in isolation is possible precisely because we kept the stages as separate functions.

In [ ]:
%%writefile test_pipeline.py
import pandas as pd

# The transform logic, copied here so the test is self-contained.
# In the assembled project it would be imported from the pipeline module.
def transform(sales_df, products):
    df = sales_df.copy()
    df["region"] = df["region"].str.strip().str.title()
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
    df = df.dropna(subset=["quantity", "unit_price"]).copy()
    df["product_name"] = df["product_id"].map(lambda pid: products.get(pid, {}).get("name", "UNKNOWN"))
    df["quantity"] = df["quantity"].astype(int)
    df["revenue"] = df["quantity"] * df["unit_price"]
    return df.reset_index(drop=True)

def test_drops_incomplete_rows():
    sales = pd.DataFrame({
        "product_id": ["P1", "P1"],
        "quantity": ["2", ""],          # second row has no quantity
        "unit_price": ["1.50", "1.50"],
        "region": ["North", "North"],
    })
    result = transform(sales, {"P1": {"name": "Pen"}})
    assert len(result) == 1             # the incomplete row was dropped

def test_computes_revenue():
    sales = pd.DataFrame({
        "product_id": ["P1"], "quantity": ["3"], "unit_price": ["2.00"], "region": ["north"],
    })
    result = transform(sales, {"P1": {"name": "Pen"}})
    assert result.loc[0, "revenue"] == 6.0
    assert result.loc[0, "region"] == "North"   # normalised

def test_flags_unknown_product():
    sales = pd.DataFrame({
        "product_id": ["P9"], "quantity": ["1"], "unit_price": ["5.00"], "region": ["East"],
    })
    result = transform(sales, {"P1": {"name": "Pen"}})
    assert result.loc[0, "product_name"] == "UNKNOWN"

In [ ]:
# Run the test suite against the cleaning logic.
!python -m pytest test_pipeline.py -q

## Assembling the Project

A capstone should leave you with a real project, not just notebook cells. The cell below writes the pipeline as a proper module with a command-line entry point, plus the test file, into a project folder. You can run it as a script: `python pipeline.py`. This is the same code developed above, organised for reuse.

In [ ]:
import os, textwrap

os.makedirs("data_pipeline_project", exist_ok=True)

pipeline_module = textwrap.dedent("""
    # A small ETL data pipeline: CSV + JSON -> clean -> SQLite -> report.
    import argparse
    import json
    import logging
    import sqlite3
    import pandas as pd

    logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
    log = logging.getLogger("pipeline")

    def extract_sales(csv_path):
        df = pd.read_csv(csv_path, dtype=str)
        log.info(f"extracted {len(df)} sales rows")
        return df

    def extract_products(json_path):
        with open(json_path) as f:
            return json.load(f)

    def transform(sales_df, products):
        df = sales_df.copy()
        df["region"] = df["region"].str.strip().str.title()
        df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
        df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
        df = df.dropna(subset=["quantity", "unit_price"]).copy()
        df["product_name"] = df["product_id"].map(lambda p: products.get(p, {}).get("name", "UNKNOWN"))
        df["category"] = df["product_id"].map(lambda p: products.get(p, {}).get("category", "UNKNOWN"))
        df["quantity"] = df["quantity"].astype(int)
        df["revenue"] = df["quantity"] * df["unit_price"]
        return df.reset_index(drop=True)

    def load(df, db_path, table="sales"):
        conn = sqlite3.connect(db_path)
        df.to_sql(table, conn, if_exists="replace", index=False)
        conn.close()

    def report(db_path):
        conn = sqlite3.connect(db_path)
        by_region = pd.read_sql_query(
            "SELECT region, SUM(revenue) AS total_revenue, COUNT(*) AS orders "
            "FROM sales GROUP BY region ORDER BY total_revenue DESC", conn)
        conn.close()
        return by_region

    def run_pipeline(csv_path, json_path, db_path):
        sales = extract_sales(csv_path)
        products = extract_products(json_path)
        clean = transform(sales, products)
        load(clean, db_path)
        return report(db_path)

    def main():
        parser = argparse.ArgumentParser(description="Run the sales data pipeline.")
        parser.add_argument("--sales", default="sales.csv")
        parser.add_argument("--products", default="products.json")
        parser.add_argument("--db", default="warehouse.db")
        args = parser.parse_args()
        summary = run_pipeline(args.sales, args.products, args.db)
        print(summary.to_string(index=False))

    if __name__ == "__main__":
        main()
""").lstrip()

with open("data_pipeline_project/pipeline.py", "w") as f:
    f.write(pipeline_module)

readme = textwrap.dedent("""
    # Sales Data Pipeline

    An ETL pipeline: reads a sales CSV and a products JSON, cleans and joins them,
    stores the result in SQLite, and prints a revenue-by-region report.

    ## Usage
    ```
    python pipeline.py --sales sales.csv --products products.json --db warehouse.db
    ```

    ## Stages
    - extract: read CSV (Pandas) and JSON (stdlib)
    - transform: normalise text, coerce numbers, drop incomplete rows, join products, derive revenue
    - load: write to a SQLite table
    - report: aggregate revenue by region
""").lstrip()
with open("data_pipeline_project/README.md", "w") as f:
    f.write(readme)

print("project assembled:")
for name in sorted(os.listdir("data_pipeline_project")):
    print("  data_pipeline_project/" + name)

In [ ]:
# Verify the assembled project runs as a standalone script on the sample data.
import shutil
shutil.copy("pipeline_data/sales.csv", "data_pipeline_project/sales.csv")
shutil.copy("pipeline_data/products.json", "data_pipeline_project/products.json")
import subprocess, sys
result = subprocess.run(
    [sys.executable, "pipeline.py"],
    cwd="data_pipeline_project", capture_output=True, text=True,
)
print("script output:")
print(result.stdout)
print("the assembled project runs end to end as a real command-line tool")

---

## Demonstrated Variations

Below are a few variations that show how the pipeline flexes, each building on the runnable code above.

### Variation 1 — Handling a completely empty input

In [ ]:
import pandas as pd
# transform should cope gracefully with no rows, returning an empty frame.
empty = pd.DataFrame(columns=["product_id", "quantity", "unit_price", "region"])
result = transform(empty, products)
print("rows out of empty input:", len(result))

### Variation 2 — A different report: top products by revenue

In [ ]:
import sqlite3, pandas as pd
conn = sqlite3.connect("pipeline_data/warehouse.db")
top_products = pd.read_sql_query(
    "SELECT product_name, SUM(revenue) AS revenue, SUM(quantity) AS units "
    "FROM sales GROUP BY product_name ORDER BY revenue DESC", conn)
conn.close()
print(top_products.to_string(index=False))

### Variation 3 — Adding a derived column in transform

In [ ]:
import pandas as pd
# Re-run transform and add a high-value flag for orders over a threshold.
df = transform(sales_df, products)
df["high_value"] = df["revenue"] >= 6.0
print(df[["order_id", "product_name", "revenue", "high_value"]].to_string(index=False))

---

## Extension Challenges

Take the project further. Solutions follow.

**Challenge 1.** Add a transform rule that rejects rows with a negative `quantity` or `unit_price` (treat them as invalid, like missing values). Demonstrate with a small input containing a negative quantity.

In [ ]:
# Your solution here


**Challenge 2.** Write a new report function that returns, for each region, the single product with the highest revenue in that region. (Hint: SQL with a grouped maximum, or Pandas `groupby` then `idxmax`.)

In [ ]:
# Your solution here


**Challenge 3.** Add a `validate_extract` step that raises a clear error if the sales CSV is missing any required column, and demonstrate it catching a malformed input.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import pandas as pd

def transform_reject_negatives(sales_df, products):
    df = sales_df.copy()
    df["region"] = df["region"].str.strip().str.title()
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
    # Treat negatives as invalid by setting them to NaN, then drop.
    df.loc[df["quantity"] < 0, "quantity"] = pd.NA
    df.loc[df["unit_price"] < 0, "unit_price"] = pd.NA
    df = df.dropna(subset=["quantity", "unit_price"]).copy()
    df["quantity"] = df["quantity"].astype(int)
    df["revenue"] = df["quantity"] * df["unit_price"]
    return df.reset_index(drop=True)

test = pd.DataFrame({
    "product_id": ["P1", "P1"], "quantity": ["3", "-2"],
    "unit_price": ["2.0", "2.0"], "region": ["North", "North"],
})
print("rows kept:", len(transform_reject_negatives(test, products)), "(the negative row was rejected)")

**Solution 2**

In [ ]:
import sqlite3, pandas as pd
conn = sqlite3.connect("pipeline_data/warehouse.db")
df = pd.read_sql_query("SELECT region, product_name, SUM(revenue) AS revenue "
                       "FROM sales GROUP BY region, product_name", conn)
conn.close()
# For each region, the row with the maximum revenue.
top_per_region = df.loc[df.groupby("region")["revenue"].idxmax()]
print(top_per_region.to_string(index=False))

**Solution 3**

In [ ]:
import pandas as pd

REQUIRED = {"order_id", "product_id", "quantity", "unit_price", "region"}

def validate_extract(df):
    missing = REQUIRED - set(df.columns)
    if missing:
        raise ValueError(f"sales data is missing required columns: {sorted(missing)}")
    return df

good = pd.DataFrame(columns=list(REQUIRED))
print("valid frame passes:", validate_extract(good) is good)

bad = pd.DataFrame(columns=["order_id", "product_id"])   # missing several columns
try:
    validate_extract(bad)
except ValueError as e:
    print("caught:", e)

---

## Summary and Key Points

- A pipeline is built as separate, single-purpose stages, extract, transform, load, report, composed into one entry point; separation makes each stage testable and the data-quality rules auditable.
- Extraction reads raw sources (CSV via Pandas, JSON via the standard library) and is a natural place to attach logging so each run leaves a trace.
- Transformation does the real work: normalising text, coercing types with `pd.to_numeric(..., errors="coerce")`, dropping or flagging bad rows explicitly, joining sources, and deriving fields.
- Loading persists the clean data to SQLite (`DataFrame.to_sql`), after which it can be queried with SQL or read back with Pandas for the report.
- The riskiest stage (transform) is tested in isolation with pytest, and the whole project is assembled into a runnable command-line script with a README, a real deliverable rather than loose cells.

### Next capstone: 9.2 — An Async API Aggregator

The next capstone fetches data from several services concurrently with `asyncio` and `httpx`, merges the results, and adds caching and retries, integrating Track 5 with the web skills of Track 8.